Build a sentiment analysis model using a pre-trained transformer by fine-tuning models like BERT, RoBERTa, or DistilBERT on domain-specific data, incorporating transfer learning, and optimizing inference efficiency for real-time applications.

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np


In [3]:
import pandas as pd
import os

# Load the saved final dataset
df = pd.read_pickle("saved_data/womens_clothing_final.pkl")

print("Dataset loaded successfully!")
print("Dataset Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Dataset loaded successfully!
Dataset Shape: (23483, 49)

Columns:
['product_id', 'category', 'subcategory1', 'subcategory2', 'location', 'channel', 'customer_age', 'review_title', 'review_text', 'rating', 'recommend_flag', 'vader_negative', 'vader_neutral', 'vader_positive', 'vader_compound', 'vader_sentiment_score', 'vader_sentiment_label', 'age_cluster_raw', 'age_group', 'sentiment_target', 'text_clean', 'text_stemmed', 'text_lemmatized', 'review_title_binned', 'category_General', 'category_General Petite', 'category_Initmates', 'subcategory1_Bottoms', 'subcategory1_Dresses', 'subcategory1_Intimate', 'subcategory1_Jackets', 'subcategory1_Tops', 'subcategory1_Trend', 'location_Bangalore', 'location_Chennai', 'location_Gurgaon', 'location_Mumbai', 'channel_Mobile', 'channel_Web', 'vader_sentiment_label_Negative', 'vader_sentiment_label_Neutral', 'vader_sentiment_label_Positive', 'age_group_Age Group 1', 'age_group_Age Group 2', 'age_group_Age Group 3', 'age_group_Age Group 4', 'text_ne

Create the sentiment dataset

In [4]:
# Create a separate dataset for transformer sentiment analysis
sentiment_df = df[["review_text", "rating"]].copy()

# Remove missing and empty reviews
sentiment_df.dropna(subset=["review_text", "rating"], inplace=True)

sentiment_df["review_text"] = (
    sentiment_df["review_text"].astype(str).str.strip()
)

sentiment_df = sentiment_df[
    sentiment_df["review_text"] != ""
].copy()

# Create sentiment labels from ratings
def assign_sentiment(rating):
    if rating <= 2:
        return 0       # Negative
    elif rating == 3:
        return 1       # Neutral
    else:
        return 2       # Positive

sentiment_df["sentiment"] = sentiment_df["rating"].apply(
    assign_sentiment
)

# Display results
print("Dataset Shape:", sentiment_df.shape)

print("\nSentiment Distribution:")
print(sentiment_df["sentiment"].value_counts().sort_index())

print("\nSample Reviews:")
display(sentiment_df.head(10))

Dataset Shape: (23483, 3)

Sentiment Distribution:
sentiment
0     2407
1     2871
2    18205
Name: count, dtype: int64

Sample Reviews:


,review_text,rating,sentiment
0,Absolutely wonderful - silky and sexy and comfortable,4,2
1,"Love this dress! it's sooo pretty. i happened to find it in a store, and i'm glad i did bc i never would have ordered it online bc it's petite. i bought a petite and am 5'8"". i love the length on me- hits just a little below the knee. would definitely be a true midi on someone who is truly petite.",5,2
2,"I had such high hopes for this dress and really wanted it to work for me. i initially ordered the petite small (my usual size) but i found this to be outrageously small. so small in fact that i could not zip it up! i reordered it in petite medium, which was just ok. overall, the top half was comfortable and fit nicely, but the bottom half had a very tight under layer and several somewhat cheap (net) over layers. imo, a major design flaw was the net over layer sewn directly into the zipper - it c",3,1
3,"I love, love, love this jumpsuit. it's fun, flirty, and fabulous! every time i wear it, i get nothing but great compliments!",5,2
4,This shirt is very flattering to all due to the adjustable front tie. it is the perfect length to wear with leggings and it is sleeveless so it pairs well with any cardigan. love this shirt!!!,5,2
5,"I love tracy reese dresses, but this one is not for the very petite. i am just under 5 feet tall and usually wear a 0p in this brand. this dress was very pretty out of the package but its a lot of dress. the skirt is long and very full so it overwhelmed my small frame. not a stranger to alterations, shortening and narrowing the skirt would take away from the embellishment of the garment. i love the color and the idea of the style but it just did not work on me. i returned this dress.",2,0
6,"I aded this in my basket at hte last mintue to see what it would look like in person. (store pick up). i went with teh darkler color only because i am so pale :-) hte color is really gorgeous, and turns out it mathced everythiing i was trying on with it prefectly. it is a little baggy on me and hte xs is hte msallet size (bummer, no petite). i decided to jkeep it though, because as i said, it matvehd everything. my ejans, pants, and the 3 skirts i waas trying on (of which i ]kept all ) oops.",5,2
7,"I ordered this in carbon for store pick up, and had a ton of stuff (as always) to try on and used this top to pair (skirts and pants). everything went with it. the color is really nice charcoal with shimmer, and went well with pencil skirts, flare pants, etc. my only compaint is it is a bit big, sleeves are long and it doesn't go in petite. also a bit loose for me, but no xxs... so i kept it and wil ldecide later since the light color is already sold out in hte smallest size...",4,2
8,I love this dress. i usually get an xs but it runs a little snug in bust so i ordered up a size. very flattering and feminine with the usual retailer flair for style.,5,2
9,"I'm 5""5' and 125 lbs. i ordered the s petite to make sure the length wasn't too long. i typically wear an xs regular in retailer dresses. if you're less busty (34b cup or smaller), a s petite will fit you perfectly (snug, but not tight). i love that i could dress it up for a party, or down for work. i love that the tulle is longer then the fabric underneath.",5,2


Split the dataset

In [5]:
from sklearn.model_selection import train_test_split

# Separate input text and target labels
x = sentiment_df["review_text"]
y = sentiment_df["sentiment"]

# First split: 80% training, 20% temporary
x_train, x_temp, y_train, y_temp = train_test_split(
    x,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Second split: 10% validation, 10% testing
x_val, x_test, y_val, y_test = train_test_split(
    x_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

# Display split sizes
print("Training Samples:", len(x_train))
print("Validation Samples:", len(x_val))
print("Testing Samples:", len(x_test))

print("\nTraining Distribution:")
print(y_train.value_counts(normalize=True).sort_index().round(4))

print("\nValidation Distribution:")
print(y_val.value_counts(normalize=True).sort_index().round(4))

print("\nTesting Distribution:")
print(y_test.value_counts(normalize=True).sort_index().round(4))

Training Samples: 18786
Validation Samples: 2348
Testing Samples: 2349

Training Distribution:
sentiment
0    0.1025
1    0.1223
2    0.7753
Name: proportion, dtype: float64

Validation Distribution:
sentiment
0    0.1026
1    0.1222
2    0.7751
Name: proportion, dtype: float64

Testing Distribution:
sentiment
0    0.1026
1    0.1222
2    0.7752
Name: proportion, dtype: float64


In [6]:
%pip install -U transformers datasets accelerate evaluate scikit-learn torch

  Using cached click-8.5.0-py3-none-any.whl.metadata (2.6 kB)
  Using cached idna-3.20-py3-none-any.whl.metadata (7.2 kB)
  Using cached fsspec-2026.7.0-py3-none-any.whl.metadata (10 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
   ---------------------------------------- 0.0/13.1 MB ? eta -:--:--
    --------------------------------------- 0.3/13.1 MB ? eta -:--:--
   - -------------------------------------- 0.5/13.1 MB 1.7 MB/s eta 0:00:08
   --- ------------------------------------ 1.0/13.1 MB 1.7 MB/s eta 0:00:07
   ---- ----------------------------------- 1.3/13.1 MB 1.8 MB/s eta 0:00:07
   ---- ----------------------------------- 1.3/13.1 MB 1.8 MB/s eta 0:00:07
   ---- ----------------------------------- 1.6/13.1 MB 1.3 MB/s eta 0:00:09
   ---- ----------------------------------- 1.6/13.1 MB 1.3 MB/s eta 0:00:09
   ----- ---------------------------------- 1.8/13.1 MB 1.2 MB/s eta 0:00:10
   ------

  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.

[notice] A new release of pip is available: 26.1.2 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import torch
import transformers
import sklearn

In [5]:
print("PyTorch Version:", torch.__version__)
print("Transformers Version:", transformers.__version__)
print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU")

PyTorch Version: 2.15.0+cpu
Transformers Version: 5.22.1
CUDA Available: False
Running on CPU
